# Deployment Optimization and Benchmarking

This notebook implements the complete deployment optimization pipeline:
1. **Model Selection** - Analyze ablation results to identify optimal variant
2. **Create Optimal Variant** - Generate PPC 5×5 variant model
3. **Export Models** - Export to ONNX, OpenVINO INT8, TensorRT FP16/INT8
4. **Benchmark** - Comprehensive benchmarking of all variants
5. **Visualize** - Generate comparison charts and analysis
6. **Report** - Create deployment summary report

## Optimal Model: PPC 5×5
- **FPS**: 182.72 (vs baseline 171.78, +6.4% improvement)
- **Latency**: 5.47ms (vs baseline 5.82ms, 6.0% faster)
- **Accuracy**: Same mAP50 and mAP50-95 as baseline

## Setup and Imports

In [1]:
# Setup and Imports
import sys
import os
import glob
import time
import gc
import json
from pathlib import Path

# Add mamba environment to Python path
mamba_env_path = '/home/tiehangz/micromamba/envs/yolov12'
if os.path.exists(mamba_env_path):
    # Find Python version dynamically
    python_lib_pattern = f'{mamba_env_path}/lib/python*/site-packages'
    python_lib_dirs = glob.glob(python_lib_pattern)
    if python_lib_dirs:
        mamba_site_packages = python_lib_dirs[0]
        if mamba_site_packages not in sys.path:
            sys.path.insert(0, mamba_site_packages)
            print(f"✓ Added mamba environment to Python path: {mamba_site_packages}")
    else:
        # Fallback: try common Python versions
        for py_version in ['3.11', '3.10', '3.9', '3.8']:
            mamba_site_packages = f'{mamba_env_path}/lib/python{py_version}/site-packages'
            if os.path.exists(mamba_site_packages) and mamba_site_packages not in sys.path:
                sys.path.insert(0, mamba_site_packages)
                print(f"✓ Added mamba environment to Python path: {mamba_site_packages}")
                break
else:
    print(f"⚠ Mamba environment not found at: {mamba_env_path}")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import rcParams
import seaborn as sns

# Ultralytics imports
from ultralytics import YOLO
from ultralytics.nn.modules.block import (
    Attention, AttentionNoPE, AttentionRPE, AttentionRoPE,
    AttentionPPC3x3, AttentionPPC5x5, AttentionPPC7x7, AttentionPPC9x9,
    PSABlock, PSA, C2PSA, C2fPSA
)
import torch.nn as nn

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
rcParams['figure.figsize'] = (12, 8)
rcParams['font.size'] = 11
sns.set_palette("husl")

# Device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cuda_available = torch.cuda.is_available()
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    print(f"CUDA Available: {cuda_available}")
    print(f"Device: {device_name}")
else:
    print("CUDA Not Available - Running on CPU")
    device_name = "CPU"

# Output directory
output_dir = Path('/home/tiehangz/proj/yolov12/modification/outputs')
output_dir.mkdir(exist_ok=True, parents=True)

print(f"\nPyTorch Version: {torch.__version__}")
print(f"Output directory: {output_dir}")



⚠ Mamba environment not found at: /home/tiehangz/micromamba/envs/yolov12
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.


ImportError: cannot import name 'AttentionPPC3x3' from 'ultralytics.nn.modules.block' (/home/tiehangz/proj/yolov12/ultralytics/nn/modules/block.py)

## 1. Model Selection Analysis

Load and analyze ablation results to identify the optimal model variant.



In [ ]:
# Load and analyze ablation results
csv_path = output_dir / 'comprehensive_comprehensive_pe_ablation_results.csv'

if not csv_path.exists():
    print(f"⚠ Warning: Results CSV not found at {csv_path}")
    print("Please run the ablation study first.")
else:
    df = pd.read_csv(csv_path)
    
    print("=" * 80)
    print("COMPREHENSIVE ABLATION RESULTS ANALYSIS")
    print("=" * 80)
    
    # Filter GPU results
    gpu_results = df[df['device'] == 'cuda'].copy()
    
    if len(gpu_results) == 0:
        print("Warning: No GPU results found. Using all results.")
        gpu_results = df.copy()
    
    print("\nAll Variants (sorted by FPS):")
    display_cols = ['variant_label', 'fps', 'avg_latency_ms', 'map50', 'map50_95', 'max_memory_mb']
    available_cols = [col for col in display_cols if col in gpu_results.columns]
    print(gpu_results[available_cols].sort_values('fps', ascending=False).to_string(index=False))
    
    # Find baseline
    baseline = gpu_results[gpu_results['variant_type'] == 'baseline'].iloc[0]
    
    # Find best variant
    best_idx = gpu_results['fps'].idxmax()
    best = gpu_results.loc[best_idx]
    
    print("\n" + "=" * 80)
    print("OPTIMAL MODEL SELECTION")
    print("=" * 80)
    
    print(f"\nBaseline Model: {baseline['variant_label']}")
    print(f"  FPS: {baseline['fps']:.2f}")
    print(f"  Latency: {baseline['avg_latency_ms']:.2f} ms")
    print(f"  mAP50: {baseline['map50']:.4f}")
    print(f"  mAP50-95: {baseline['map50_95']:.4f}")
    
    print(f"\nOptimal Variant: {best['variant_label']}")
    print(f"  Variant Type: {best['variant_type']}")
    print(f"  Variant Value: {best['variant_value']}")
    print(f"  FPS: {best['fps']:.2f}")
    print(f"  Latency: {best['avg_latency_ms']:.2f} ms")
    print(f"  mAP50: {best['map50']:.4f}")
    print(f"  mAP50-95: {best['map50_95']:.4f}")
    
    fps_improvement = best['fps'] - baseline['fps']
    fps_improvement_pct = ((best['fps'] / baseline['fps'] - 1) * 100)
    latency_improvement = baseline['avg_latency_ms'] - best['avg_latency_ms']
    latency_improvement_pct = ((1 - best['avg_latency_ms'] / baseline['avg_latency_ms']) * 100)
    
    print(f"\nImprovement over Baseline:")
    print(f"  FPS: +{fps_improvement:.2f} ({fps_improvement_pct:+.1f}%)")
    print(f"  Latency: {latency_improvement:.2f} ms faster ({latency_improvement_pct:+.1f}%)")
    
    # Save selection
    selection = {
        'baseline': {
            'variant_label': baseline['variant_label'],
            'variant_type': baseline.get('variant_type', 'baseline'),
            'variant_value': baseline.get('variant_value', 'ppc_7x7'),
            'model_path': baseline['model_path'],
            'fps': float(baseline['fps']),
            'latency_ms': float(baseline['avg_latency_ms']),
            'map50': float(baseline['map50']),
            'map50_95': float(baseline['map50_95'])
        },
        'optimal': {
            'variant_label': best['variant_label'],
            'variant_type': best['variant_type'],
            'variant_value': best['variant_value'],
            'model_path': best['model_path'],
            'fps': float(best['fps']),
            'latency_ms': float(best['avg_latency_ms']),
            'map50': float(best['map50']),
            'map50_95': float(best['map50_95'])
        },
        'improvements': {
            'fps_improvement': float(fps_improvement),
            'fps_improvement_pct': float(fps_improvement_pct),
            'latency_improvement_ms': float(latency_improvement),
            'latency_improvement_pct': float(latency_improvement_pct)
        }
    }
    
    selection_file = output_dir / 'optimal_model_selection.json'
    with open(selection_file, 'w') as f:
        json.dump(selection, f, indent=2)
    
    print(f"\n✓ Selection saved to: {selection_file}")
    
    # Store for later use
    OPTIMAL_VARIANT_TYPE = best['variant_type']
    OPTIMAL_VARIANT_VALUE = best['variant_value']



COMPREHENSIVE ABLATION RESULTS ANALYSIS

All Variants (sorted by FPS):
     variant_label        fps  avg_latency_ms    map50  map50_95  max_memory_mb
           PPC 5×5 182.716935        5.472946 0.618059  0.412843      89.876465
               RPE 181.967441        5.495489 0.618059  0.412843      89.876465
              RoPE 179.212312        5.579974 0.618059  0.412843      89.876465
           PPC 9×9 177.422617        5.636260 0.618059  0.412843      89.876465
           PPC 3×3 174.439296        5.732653 0.618059  0.412843      89.876465
Baseline (PPC 7×7) 171.778611        5.821447 0.618059  0.412843      74.933594
              NoPE 171.357913        5.835739 0.618059  0.412843      91.830078

OPTIMAL MODEL SELECTION

Baseline Model: Baseline (PPC 7×7)
  FPS: 171.78
  Latency: 5.82 ms
  mAP50: 0.6181
  mAP50-95: 0.4128

Optimal Variant: PPC 5×5
  Variant Type: ppc
  Variant Value: ppc_5x5
  FPS: 182.72
  Latency: 5.47 ms
  mAP50: 0.6181
  mAP50-95: 0.4128

Improvement over Bas

## 2. Helper Function: Replace Attention Modules

In [ ]:
def replace_attention_modules(model, variant_type='ppc', variant_value=7):
    """Replace all Attention modules in a model with specified variant."""
    replaced_modules = []
    
    if variant_type == 'ppc':
        variant_map = {3: AttentionPPC3x3, 5: AttentionPPC5x5, 7: AttentionPPC7x7, 9: AttentionPPC9x9}
        if variant_value not in variant_map:
            raise ValueError(f"Unknown PPC kernel size: {variant_value}")
        new_class = variant_map[variant_value]
        variant_name = f"PPC {variant_value}x{variant_value}"
    elif variant_type == 'pe':
        variant_map = {'none': AttentionNoPE, 'rpe': AttentionRPE, 'rope': AttentionRoPE}
        if variant_value not in variant_map:
            raise ValueError(f"Unknown PE variant: {variant_value}")
        new_class = variant_map[variant_value]
        variant_name = variant_value.upper()
    else:
        raise ValueError(f"Unknown variant_type: {variant_type}")
    
    def _replace_module(module, name=''):
        for child_name, child_module in list(module.named_children()):
            full_name = f"{name}.{child_name}" if name else child_name
            
            if isinstance(child_module, Attention):
                dim = child_module.head_dim * child_module.num_heads
                num_heads = child_module.num_heads
                attn_ratio = child_module.key_dim / child_module.head_dim
                
                if variant_type == 'ppc':
                    new_module = new_class(dim, num_heads, attn_ratio)
                elif variant_type == 'pe':
                    if variant_value == 'rope':
                        new_module = new_class(dim, num_heads, attn_ratio, rope_theta=10000.0)
                    else:
                        new_module = new_class(dim, num_heads, attn_ratio)
                
                try:
                    if hasattr(child_module, 'qkv') and hasattr(new_module, 'qkv'):
                        new_module.qkv.weight.data.copy_(child_module.qkv.weight.data)
                    if hasattr(child_module, 'proj') and hasattr(new_module, 'proj'):
                        new_module.proj.weight.data.copy_(child_module.proj.weight.data)
                    if variant_type == 'ppc' and hasattr(child_module, 'ppc') and hasattr(new_module, 'ppc'):
                        if child_module.ppc_kernel_size == new_module.ppc_kernel_size:
                            new_module.ppc.weight.data.copy_(child_module.ppc.weight.data)
                except Exception as e:
                    pass
                
                setattr(module, child_name, new_module)
                replaced_modules.append((full_name, child_module, new_module))
            
            elif isinstance(child_module, (PSABlock, PSA, C2PSA, C2fPSA, nn.Sequential, nn.ModuleList)):
                _replace_module(child_module, full_name)
            else:
                if len(list(child_module.children())) > 0:
                    _replace_module(child_module, full_name)
    
    if hasattr(model, 'model'):
        _replace_module(model.model)
    else:
        _replace_module(model)
    
    return replaced_modules

print("✓ Helper function defined")

✓ Helper function defined


## 3. Create Optimal Variant Model

In [ ]:
# Create optimal PPC 5×5 variant
baseline_path = Path('/home/tiehangz/proj/yolov12/model/yolov12n-seg.pt')
optimal_path = Path('/home/tiehangz/proj/yolov12/modification/yolov12n-seg-ppc5x5.pt')

if not baseline_path.exists():
    print(f"⚠ Warning: Baseline model not found at {baseline_path}")
else:
    print(f"Loading baseline model: {baseline_path}")
    model = YOLO(str(baseline_path))
    
    print(f"\nApplying PPC 5×5 modification...")
    replaced = replace_attention_modules(model, variant_type='ppc', variant_value=5)
    
    if len(replaced) == 0:
        print("Warning: No Attention modules found to replace!")
    else:
        print(f"Successfully replaced {len(replaced)} Attention module(s)")
    
    print(f"\nSaving optimal variant to: {optimal_path}")
    model.save(str(optimal_path))
    
    if optimal_path.exists():
        file_size_mb = optimal_path.stat().st_size / (1024 * 1024)
        print(f"✓ Model saved successfully ({file_size_mb:.2f} MB)")
    else:
        print("✗ Failed to save model")

Loading baseline model: /home/tiehangz/proj/yolov12/model/yolov12n-seg.pt

Applying PPC 5×5 modification...

Saving optimal variant to: /home/tiehangz/proj/yolov12/modification/yolov12n-seg-ppc5x5.pt
✓ Model saved successfully (5.78 MB)


## 4. Export Models to Optimized Formats

In [ ]:
# Export models to various formats
baseline_path = Path('/home/tiehangz/proj/yolov12/model/yolov12n-seg.pt')
optimal_path = Path('/home/tiehangz/proj/yolov12/modification/yolov12n-seg-ppc5x5.pt')
calibration_data = 'coco128-seg.yaml'

models_to_export = [('baseline', baseline_path), ('optimal', optimal_path)]
export_results = {}

for model_name, model_path in models_to_export:
    if not model_path.exists():
        print(f"\n⚠ Warning: {model_name} model not found at {model_path}")
        continue
    
    print(f"\n{'=' * 80}")
    print(f"Processing {model_name.upper()} model")
    print("=" * 80)
    
    model = YOLO(str(model_path))
    model_results = {}
    
    # Export to ONNX
    print(f"\n[1/4] Exporting {model_name} to ONNX...")
    try:
        onnx_path = model.export(format='onnx', simplify=True, opset=17, imgsz=640, dynamic=False)
        model_results['onnx'] = str(onnx_path)
        print(f"✓ ONNX export successful")
    except Exception as e:
        print(f"✗ ONNX export failed: {e}")
        model_results['onnx'] = None
    
    # Export to OpenVINO INT8
    print(f"\n[2/4] Exporting {model_name} to OpenVINO INT8...")
    try:
        openvino_path = model.export(format='openvino', int8=True, data=calibration_data, imgsz=640)
        model_results['openvino_int8'] = str(openvino_path)
        print(f"✓ OpenVINO INT8 export successful")
    except Exception as e:
        print(f"✗ OpenVINO INT8 export failed: {e}")
        model_results['openvino_int8'] = None
    
    # Export to TensorRT FP16
    print(f"\n[3/4] Exporting {model_name} to TensorRT FP16...")
    try:
        trt_fp16_path = model.export(format='engine', half=True, imgsz=640, workspace=4)
        model_results['tensorrt_fp16'] = str(trt_fp16_path)
        print(f"✓ TensorRT FP16 export successful")
    except Exception as e:
        print(f"✗ TensorRT FP16 export failed: {e}")
        model_results['tensorrt_fp16'] = None
    
    # Export to TensorRT INT8
    print(f"\n[4/4] Exporting {model_name} to TensorRT INT8...")
    try:
        trt_int8_path = model.export(format='engine', int8=True, data=calibration_data, imgsz=640, workspace=4)
        model_results['tensorrt_int8'] = str(trt_int8_path)
        print(f"✓ TensorRT INT8 export successful")
    except Exception as e:
        print(f"✗ TensorRT INT8 export failed: {e}")
        model_results['tensorrt_int8'] = None
    
    export_results[model_name] = model_results

# Save export results
export_results_file = output_dir / 'export_results.json'
with open(export_results_file, 'w') as f:
    json.dump(export_results, f, indent=2)

print(f"\n✓ Export results saved to: {export_results_file}")


Processing BASELINE model

[1/4] Exporting baseline to ONNX...
Ultralytics 8.3.63 🚀 Python-3.11.14 torch-2.9.0+cu128 CPU (AMD Ryzen 9 7945HX with Radeon Graphics)
YOLOv12n-seg summary (fused): 403 layers, 2,795,240 parameters, 0 gradients, 9.2 GFLOPs

PyTorch: starting from '/home/tiehangz/proj/yolov12/model/yolov12n-seg.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 116, 8400), (1, 32, 160, 160)) (5.8 MB)

ONNX: starting export with onnx 1.19.1 opset 17...


W1111 17:11:28.735000 25912 site-packages/torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/tiehangz/micromamba/envs/yolov12/lib/python3.11/site-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c

Applied 9 of general pattern rewrite rules.
ONNX: slimming with onnxslim 0.1.74...
ONNX: export success ✅ 4.0s, saved as '/home/tiehangz/proj/yolov12/model/yolov12n-seg.onnx' (10.9 MB)

Export complete (4.3s)
Results saved to /home/tiehangz/proj/yolov12/model
Predict:         yolo predict task=segment model=/home/tiehangz/proj/yolov12/model/yolov12n-seg.onnx imgsz=640  
Validate:        yolo val task=segment model=/home/tiehangz/proj/yolov12/model/yolov12n-seg.onnx imgsz=640 data=/data_local2/tianyunjie/slurm_projects/yolov12-seg/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app
✓ ONNX export successful

[2/4] Exporting baseline to OpenVINO INT8...
Ultralytics 8.3.63 🚀 Python-3.11.14 torch-2.9.0+cu128 CPU (AMD Ryzen 9 7945HX with Radeon Graphics)
YOLOv12n-seg summary (fused): 403 layers, 2,795,240 parameters, 0 gradients, 9.2 GFLOPs

PyTorch: starting from '/home/tiehangz/proj/yolov12/model/yolov12n-seg.pt' with input shape (1, 3, 640, 640) BCHW and output shape(

Scanning /home/tiehangz/proj/datasets/coco128-seg/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]

OpenVINO: WARNING ⚠️ >300 images recommended for INT8 calibration, found 128 images.


...

OpenVINO: export success ✅ 75.7s, saved as '/home/tiehangz/proj/yolov12/modification/yolov12n-seg-ppc5x5_int8_openvino_model/' (3.9 MB)

Export complete (76.9s)
Results saved to /home/tiehangz/proj/yolov12/modification
Predict:         yolo predict task=segment model=/home/tiehangz/proj/yolov12/modification/yolov12n-seg-ppc5x5_int8_openvino_model imgsz=640 int8 
Validate:        yolo val task=segment model=/home/tiehangz/proj/yolov12/modification/yolov12n-seg-ppc5x5_int8_openvino_model imgsz=640 data=/data_local2/tianyunjie/slurm_projects/yolov12-seg/ultralytics/cfg/datasets/coco.yaml int8 
Visualize:       https://netron.app
✓ OpenVINO INT8 export successful

[3/4] Exporting optimal to TensorRT FP16...
WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.3.63 🚀 Python-3.11.14 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
YOLOv12n-seg summary (fused): 403 layers, 2,795,240 parameters, 0 gradients, 9.2 GFLOPs

PyTorch: starting

W1111 17:17:36.799000 25912 site-packages/torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 10 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 10).
Failed to convert the model to the target version 10 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/tiehangz/micromamba/envs/yolov12/lib/python3.11/site-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c

Applied 9 of general pattern rewrite rules.
ONNX: slimming with onnxslim 0.1.74...
ONNX: export success ✅ 4.3s, saved as '/home/tiehangz/proj/yolov12/modification/yolov12n-seg-ppc5x5.onnx' (10.9 MB)

TensorRT: starting export with TensorRT 10.14.1.48.post1...
[11/11/2025-17:17:41] [TRT] [I] ----------------------------------------------------------------
[11/11/2025-17:17:41] [TRT] [I] Input filename:   /home/tiehangz/proj/yolov12/modification/yolov12n-seg-ppc5x5.onnx
[11/11/2025-17:17:41] [TRT] [I] ONNX IR version:  0.0.10
[11/11/2025-17:17:41] [TRT] [I] Opset version:    18
[11/11/2025-17:17:41] [TRT] [I] Producer name:    pytorch
[11/11/2025-17:17:41] [TRT] [I] Producer version: 2.9.0+cu128
[11/11/2025-17:17:41] [TRT] [I] Domain:           
[11/11/2025-17:17:41] [TRT] [I] Model version:    0
[11/11/2025-17:17:41] [TRT] [I] Doc string:       
[11/11/2025-17:17:41] [TRT] [I] ----------------------------------------------------------------
TensorRT: input "images" with shape(1, 3, 640,

W1111 17:20:43.148000 25912 site-packages/torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 10 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W1111 17:20:45.736000 25912 site-packages/torch/fx/experimental/symbolic_shapes.py:6833] _maybe_guard_rel() was called on non-relation expression ((((s0 - 1)//2)) + 1 > 0) | ((((s53 - 1)//2)) + 1 > 0)
W1111 17:20:45.975000 25912 site-packages/torch/fx/experimental/symbolic_shapes.py:6833] _maybe_guard_rel() was called on non-relation expression ((((s0 - 1)//4)) + 1 > 0) | ((((s53 - 1)//4)) + 1 > 0)
W1111 17:20:46.881000 25912 site-packages/torch/fx/experimental/symbolic_s

ONNX: export failure ❌ 29.4s: Failed to decompose the FX graph for ONNX compatibility. This is step 2/3 of exporting the model to ONNX. Next steps:
- Create an issue in the PyTorch GitHub repository against the *torch.export* component and attach the full error stack as well as reproduction scripts.
- Create an error report with `torch.onnx.export(..., report=True)`, and save the ExportedProgram as a pt2 file. Create an issue in the PyTorch GitHub repository against the *onnx* component. Attach the error report and the pt2 model.

## Exception summary

<class 'AttributeError'>: 'float' object has no attribute 'node'

While executing %item : [num_users=1] = call_function[target=torch.ops.aten.item.default](args = (%getitem_38,), kwargs = {})
Original traceback:
File "/home/tiehangz/proj/yolov12/ultralytics/nn/tasks.py", line 111, in forward
    return self.predict(x, *args, **kwargs)
  File "/home/tiehangz/proj/yolov12/ultralytics/nn/modules/head.py", line 194, in forward
    x = Detect

## 5. Benchmark All Model Variants

In [ ]:
# Benchmarking function
def benchmark_pytorch_model(model_path, test_data='coco128-seg.yaml', imgsz=640, warmup=10, iterations=100, device='cuda'):
    """Benchmark PyTorch model."""
    model = YOLO(str(model_path))
    dummy_input = torch.randn(1, 3, imgsz, imgsz).to(device)
    
    # Warmup
    for _ in range(warmup):
        _ = model.predict(dummy_input, verbose=False, imgsz=imgsz)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    # Benchmark
    latencies = []
    for _ in range(iterations):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = model.predict(dummy_input, verbose=False, imgsz=imgsz)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end = time.perf_counter()
        latencies.append((end - start) * 1000)
    
    avg_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    fps = 1000.0 / avg_latency
    
    max_memory = torch.cuda.max_memory_allocated() / (1024 * 1024) if torch.cuda.is_available() else 0.0
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    
    try:
        metrics = model.val(data=test_data, imgsz=imgsz, verbose=False)
        
        # Debug: Print available attributes
        print(f'  Available seg attributes: {[attr for attr in dir(metrics.seg) if not attr.startswith("_")]}')
        if hasattr(metrics.seg, 'map'):
            print(f'  metrics.seg.map = {metrics.seg.map}')
        if hasattr(metrics.seg, 'map50'):
            print(f'  metrics.seg.map50 = {metrics.seg.map50}')
        
        # Extract mAP50
        map50 = metrics.seg.map50 if hasattr(metrics.seg, 'map50') else 0.0
        
        # Extract mAP50-95 - try multiple methods

        # Extract mAP50
        map50 = metrics.seg.map50 if hasattr(metrics.seg, 'map50') else 0.0

        # Extract mAP50-95 - try multiple methods
        map50_95 = None

        # Method 1: Direct attribute (most common)
        if hasattr(metrics.seg, 'map'):
            map50_95 = metrics.seg.map

        # Method 2: From maps array
        elif hasattr(metrics.seg, 'maps') and metrics.seg.maps is not None:
            if isinstance(metrics.seg.maps, (list, tuple)) and len(metrics.seg.maps) > 0:
                map50_95 = np.mean(metrics.seg.maps)
            elif isinstance(metrics.seg.maps, np.ndarray) and metrics.seg.maps.size > 0:
                map50_95 = float(np.mean(metrics.seg.maps))

        # Method 3: From results_dict
        if map50_95 is None and hasattr(metrics, 'results_dict'):
            results_dict = metrics.results_dict
            if isinstance(results_dict, dict):
                for key in ['metrics/mAP50-95(M)', 'mAP50-95(M)', 'map50_95', 'mask/mAP50-95']:
                    if key in results_dict:
                        map50_95 = float(results_dict[key])
                        break

        # Method 4: From all_ap if available
        if map50_95 is None and hasattr(metrics.seg, 'all_ap'):
            all_ap = metrics.seg.all_ap
            if isinstance(all_ap, np.ndarray) and all_ap.size > 0:
                # all_ap shape is (nc, 10) where 10 is IoU thresholds from 0.5 to 0.95
                map50_95 = float(all_ap.mean())
                # all_ap shape is (nc, 10) where 10 is IoU thresholds from 0.5 to 0.95
                map50_95 = float(all_ap.mean())

        # Default to 0 if still None
        if map50_95 is None:
            map50_95 = 0.0
            print(f'  ⚠️  Warning: Could not extract mAP50-95, using 0.0')
        else:
            print(f'  ✓ Extracted mAP50-95: {map50_95:.4f}')
        
        
        if map50_95 is None and hasattr(metrics, 'results_dict'):
            if isinstance(results_dict, dict):
                for key in ['metrics/mAP50-95(M)', 'mAP50-95(M)', 'map50_95', 'mask/mAP50-95']:
                    if key in results_dict:
                        break
        
        if map50_95 is None and hasattr(metrics.seg, 'all_ap'):
            if isinstance(all_ap, np.ndarray) and all_ap.size > 0:
                # all_ap shape is (nc, 10) where 10 is IoU thresholds from 0.5 to 0.95
                map50_95 = float(all_ap.mean())
                # all_ap shape is (nc, 10) where 10 is IoU thresholds from 0.5 to 0.95
        
        # Default to 0 if still None
        if map50_95 is None:
            print(f'  ⚠️  Warning: Could not extract mAP50-95, using 0.0')
        else:
            print(f'  ✓ Extracted mAP50-95: {map50_95:.4f}')
        
        # Extract mAP50
        
        # Extract mAP50-95 - try multiple methods

        # Extract mAP50
        map50 = metrics.seg.map50 if hasattr(metrics.seg, 'map50') else 0.0

        # Extract mAP50-95 - try multiple methods
        map50_95 = None

        # Method 1: Direct attribute (most common)
        if hasattr(metrics.seg, 'map'):
            map50_95 = metrics.seg.map

        # Method 2: From maps array
        elif hasattr(metrics.seg, 'maps') and metrics.seg.maps is not None:
            if isinstance(metrics.seg.maps, (list, tuple)) and len(metrics.seg.maps) > 0:
                map50_95 = np.mean(metrics.seg.maps)
            elif isinstance(metrics.seg.maps, np.ndarray) and metrics.seg.maps.size > 0:
                map50_95 = float(np.mean(metrics.seg.maps))

        # Method 3: From results_dict
        if map50_95 is None and hasattr(metrics, 'results_dict'):
            results_dict = metrics.results_dict
            if isinstance(results_dict, dict):
                for key in ['metrics/mAP50-95(M)', 'mAP50-95(M)', 'map50_95', 'mask/mAP50-95']:
                    if key in results_dict:
                        map50_95 = float(results_dict[key])
                        break

        # Method 4: From all_ap if available
        if map50_95 is None and hasattr(metrics.seg, 'all_ap'):
            all_ap = metrics.seg.all_ap
            if isinstance(all_ap, np.ndarray) and all_ap.size > 0:
                # all_ap shape is (nc, 10) where 10 is IoU thresholds from 0.5 to 0.95
                map50_95 = float(all_ap.mean())
                # all_ap shape is (nc, 10) where 10 is IoU thresholds from 0.5 to 0.95
                map50_95 = float(all_ap.mean())

        # Default to 0 if still None
        if map50_95 is None:
            map50_95 = 0.0
            print(f'  ⚠️  Warning: Could not extract mAP50-95, using 0.0')
        else:
            print(f'  ✓ Extracted mAP50-95: {map50_95:.4f}')
        
    except:
        map50 = 0.0
        map50_95 = 0.0
    
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return {'avg_latency_ms': avg_latency, 'std_latency_ms': std_latency, 'fps': fps,
            'max_memory_mb': max_memory, 'map50': map50, 'map50_95': map50_95}

print("✓ Benchmarking function defined")


IndentationError: expected an indented block after 'if' statement on line 90 (2562739993.py, line 91)

In [ ]:
# Run benchmarks
baseline_pt = Path('/home/tiehangz/proj/yolov12/model/yolov12n-seg.pt')
optimal_pt = Path('/home/tiehangz/proj/yolov12/modification/yolov12n-seg-ppc5x5.pt')
test_data = 'coco128-seg.yaml'
imgsz = 640
warmup = 10
iterations = 100

results = []

# Load export results if available
export_results_file = output_dir / 'export_results.json'
if export_results_file.exists():
    with open(export_results_file, 'r') as f:
        export_results = json.load(f)
else:
    export_results = {'baseline': {}, 'optimal': {}}

# Benchmark PyTorch models
for model_name, model_path in [('baseline', baseline_pt), ('optimal', optimal_pt)]:
    if model_path.exists():
        print(f"\n{'=' * 80}")
        print(f"Benchmarking: {model_name.upper()} - PyTorch")
        print("=" * 80)
        try:
            result = benchmark_pytorch_model(model_path, test_data, imgsz, warmup, iterations, 'cuda')
            result['model_name'] = model_name
            result['format'] = 'pytorch'
            result['device'] = 'cuda'
            result['model_path'] = str(model_path)
            result['model_size_mb'] = model_path.stat().st_size / (1024 * 1024)
            results.append(result)
            print(f"  FPS: {result['fps']:.2f}, Latency: {result['avg_latency_ms']:.2f} ms")
        except Exception as e:
            print(f"  ✗ Benchmark failed: {e}")

# Benchmark exported models
for model_name in ['baseline', 'optimal']:
    if model_name in export_results:
        for fmt, path in export_results[model_name].items():
            if path and Path(path).exists():
                print(f"\nBenchmarking: {model_name.upper()} - {fmt.upper()}")
                try:
                    result = benchmark_pytorch_model(Path(path), test_data, imgsz, warmup, iterations, 'cuda')
                    result['model_name'] = model_name
                    result['format'] = fmt
                    result['device'] = 'cuda'
                    result['model_path'] = str(path)
                    result['model_size_mb'] = Path(path).stat().st_size / (1024 * 1024)
                    results.append(result)
                    print(f"  FPS: {result['fps']:.2f}, Latency: {result['avg_latency_ms']:.2f} ms")
                except Exception as e:
                    print(f"  ✗ Benchmark failed: {e}")

# Save results
results_file = output_dir / 'deployment_benchmark_results.json'
with open(results_file, 'w') as f:
    json.dump(results, f, indent=2)

df_results = pd.DataFrame(results)

# Ensure map50_95 is numeric
if 'map50_95' in df_results.columns:
    df_results['map50_95'] = pd.to_numeric(df_results['map50_95'], errors='coerce').fillna(0.0)
    print(f"\n✓ Verified map50_95 values: {df_results['map50_95'].tolist()}")
csv_file = output_dir / 'deployment_benchmark_results.csv'
df_results.to_csv(csv_file, index=False)

print(f"\n✓ Results saved to: {results_file}")
print(f"✓ Results saved to CSV: {csv_file}")


Benchmarking: BASELINE - PyTorch
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.956011772155762. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.956011772155762. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.956011772155762. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.956011772155762. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.956011772155762. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.956011772155762. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.956011772155762. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.956011772155762. Dividing input by 255.
WARNING ⚠️ tor

val: Scanning /home/tiehangz/proj/datasets/coco128-seg/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:05<00:00,  1.44it/s]


                   all        128        929       0.63      0.596      0.652      0.491      0.607      0.571      0.622       0.41
Speed: 5.5ms preprocess, 8.3ms inference, 0.0ms loss, 3.2ms postprocess per image
Results saved to /home/tiehangz/proj/yolov12/runs/segment/val36
  FPS: 17.76, Latency: 56.31 ms

Benchmarking: OPTIMAL - PyTorch
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.718592643737793. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.718592643737793. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.718592643737793. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.718592643737793. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.718592643737793. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 b

val: Scanning /home/tiehangz/proj/datasets/coco128-seg/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.50it/s]


                   all        128        929       0.63      0.596      0.652      0.491      0.607      0.571      0.622       0.41
Speed: 2.7ms preprocess, 7.0ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /home/tiehangz/proj/yolov12/runs/segment/val37
  FPS: 124.04, Latency: 8.06 ms

Benchmarking: BASELINE - ONNX
Loading /home/tiehangz/proj/yolov12/model/yolov12n-seg.onnx for ONNX Runtime inference...
Using ONNX Runtime CUDAExecutionProvider
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.60026741027832. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.60026741027832. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.60026741027832. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.60026741027832. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0

...

                   all        128        929       0.64       0.57      0.654      0.501      0.612      0.549       0.62      0.409
Speed: 0.3ms preprocess, 4.6ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /home/tiehangz/proj/yolov12/runs/segment/val41
  FPS: 172.32, Latency: 5.80 ms

Benchmarking: OPTIMAL - OPENVINO_INT8
Loading /home/tiehangz/proj/yolov12/modification/yolov12n-seg-ppc5x5_int8_openvino_model for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference...
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.077834129333496. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.077834129333496. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.077834129333496. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.077834129333496. Dividing input by 255.
WARNIN

val: Scanning /home/tiehangz/proj/datasets/coco128-seg/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 128/128 [00:05<00:00, 23.88it/s]


                   all        128        929      0.637      0.562       0.63      0.479      0.602      0.533      0.594      0.391
Speed: 0.5ms preprocess, 19.0ms inference, 0.0ms loss, 2.2ms postprocess per image
Results saved to /home/tiehangz/proj/yolov12/runs/segment/val42
  FPS: 33.37, Latency: 29.97 ms

Benchmarking: OPTIMAL - TENSORRT_FP16
Loading /home/tiehangz/proj/yolov12/modification/yolov12n-seg-ppc5x5.engine for TensorRT inference...
[11/11/2025-16:56:48] [TRT] [I] Loaded engine size: 8 MiB
[11/11/2025-16:56:48] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +1, GPU +16, now: CPU 1, GPU 21 (MiB)
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.889733791351318. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.889733791351318. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.889733791351318. Divid

val: Scanning /home/tiehangz/proj/datasets/coco128-seg/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 128/128 [00:00<00:00, 191.00it/s]


                   all        128        929       0.64      0.575      0.655      0.501      0.611      0.553      0.619      0.411
Speed: 0.3ms preprocess, 1.6ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /home/tiehangz/proj/yolov12/runs/segment/val43
  FPS: 354.35, Latency: 2.82 ms

✓ Results saved to: /home/tiehangz/proj/yolov12/modification/outputs/deployment_benchmark_results.json
✓ Results saved to CSV: /home/tiehangz/proj/yolov12/modification/outputs/deployment_benchmark_results.csv


## 6. Create Visualizations

In [ ]:
# Load benchmark results and create visualizations
csv_file = output_dir / 'deployment_benchmark_results.csv'
if csv_file.exists():
    df = pd.read_csv(csv_file)
    
    if len(df) > 0:
        df['model_format'] = df['model_name'] + ' - ' + df['format']
        
        format_colors = {
            'pytorch': '#1f77b4',
            'onnx': '#ff7f0e',
            'openvino_int8': '#2ca02c',
            'tensorrt_fp16': '#d62728',
            'tensorrt_int8': '#9467bd'
        }
        colors = [format_colors.get(fmt, '#808080') for fmt in df['format']]
        
        # Create comprehensive figure
        fig = plt.figure(figsize=(20, 12))
        
        # FPS Comparison
        ax1 = plt.subplot(2, 3, 1)
        bars1 = ax1.barh(range(len(df)), df['fps'], color=colors, alpha=0.8)
        ax1.set_yticks(range(len(df)))
        ax1.set_yticklabels(df['model_format'], fontsize=8)
        ax1.set_xlabel('FPS', fontsize=10)
        ax1.set_title('Throughput Comparison', fontsize=12, fontweight='bold')
        ax1.axvline(x=30, color='r', linestyle='--', linewidth=2, label='30 FPS Target', alpha=0.7)
        ax1.legend()
        ax1.grid(True, alpha=0.3, axis='x')
        
        # Latency Comparison
        ax2 = plt.subplot(2, 3, 2)
        bars2 = ax2.barh(range(len(df)), df['avg_latency_ms'], color=colors, alpha=0.8)
        ax2.set_yticks(range(len(df)))
        ax2.set_yticklabels(df['model_format'], fontsize=8)
        ax2.set_xlabel('Latency (ms)', fontsize=10)
        ax2.set_title('Latency Comparison', fontsize=12, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='x')
        
        # Memory Usage
        ax3 = plt.subplot(2, 3, 3)
        memory_data = df['max_memory_mb'].fillna(0)
        bars3 = ax3.barh(range(len(df)), memory_data, color=colors, alpha=0.8)
        ax3.set_yticks(range(len(df)))
        ax3.set_yticklabels(df['model_format'], fontsize=8)
        ax3.set_xlabel('Memory (MB)', fontsize=10)
        ax3.set_title('Memory Usage', fontsize=12, fontweight='bold')
        ax3.grid(True, alpha=0.3, axis='x')
        
        # mAP50
        ax4 = plt.subplot(2, 3, 4)
        bars4 = ax4.barh(range(len(df)), df['map50'], color=colors, alpha=0.8)
        ax4.set_yticks(range(len(df)))
        ax4.set_yticklabels(df['model_format'], fontsize=8)
        ax4.set_xlabel('mAP50', fontsize=10)
        ax4.set_title('mAP50 Comparison', fontsize=12, fontweight='bold')
        ax4.grid(True, alpha=0.3, axis='x')
        
        # mAP50-95
        ax5 = plt.subplot(2, 3, 5)
        # Ensure map50_95 is numeric for visualization
        map50_95_values = pd.to_numeric(df['map50_95'], errors='coerce').fillna(0.0).values
        bars5 = ax5.barh(range(len(df)), map50_95_values, color=colors, alpha=0.8)
        ax5.set_yticks(range(len(df)))
        ax5.set_yticklabels(df['model_format'], fontsize=8)
        ax5.set_xlabel('mAP50-95', fontsize=10)
        ax5.set_title('mAP50-95 Comparison', fontsize=12, fontweight='bold')
        ax5.grid(True, alpha=0.3, axis='x')
        
        # Trade-off Plot
        ax6 = plt.subplot(2, 3, 6)
        for fmt in df['format'].unique():
            fmt_data = df[df['format'] == fmt]
            # Ensure map50_95 is numeric for scatter plot
            map50_95_vals = pd.to_numeric(fmt_data['map50_95'], errors='coerce').fillna(0.0).values
            fps_vals = pd.to_numeric(fmt_data['fps'], errors='coerce').fillna(0.0).values
            ax6.scatter(fps_vals, map50_95_vals,
                                   label=fmt.replace('_', ' ').title(),
                                   alpha=0.7, s=150,
                                   color=format_colors.get(fmt, '#808080'),
                                   edgecolors='black', linewidths=1.5)
        ax6.set_xlabel('FPS', fontsize=10)
        ax6.set_ylabel('mAP50-95', fontsize=10)
        ax6.set_title('Accuracy vs Speed Trade-off', fontsize=12, fontweight='bold')
        ax6.legend(fontsize=8)
        ax6.grid(True, alpha=0.3)
        
        plt.suptitle('Deployment Model Comparison: Baseline vs Optimal (PPC 5×5)', fontsize=14, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.99])
        
        viz_file = output_dir / 'deployment_comparison.png'
        plt.savefig(viz_file, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ Visualization saved to: {viz_file}")
        plt.show()
    else:
        print("No benchmark results available.")
else:
    print("⚠ Benchmark results not found. Please run benchmarks first.")

✓ Visualization saved to: /home/tiehangz/proj/yolov12/modification/outputs/deployment_comparison.png


<Figure size 2000x1200 with 6 Axes>

## 7. Generate Deployment Summary Report

In [ ]:
# Generate deployment summary report
report_file = output_dir / 'deployment_summary_report.md'

report_content = """# Deployment Optimization Summary Report

## Executive Summary

This report summarizes the deployment optimization process for YOLOv12n segmentation models.

## 1. Optimal Model Selection

**Optimal Variant: PPC 5×5**
- FPS: 182.72 (vs baseline 171.78, +6.4% improvement)
- Latency: 5.47ms (vs baseline 5.82ms, 6.0% faster)
- Accuracy: Same mAP50 and mAP50-95 as baseline

## 2. Benchmark Results

"""

# Add benchmark results if available
csv_file = output_dir / 'deployment_benchmark_results.csv'
if csv_file.exists():
    df_report = pd.read_csv(csv_file)
    report_content += "\n### Performance Summary\n\n"
    report_content += "| Model | Format | FPS | Latency (ms) | mAP50-95 |\n"
    report_content += "|-------|--------|-----|---------------|----------|\n"
    for _, row in df_report.iterrows():
        report_content += f"| {row['model_name']} | {row['format']} | {row['fps']:.2f} | {row['avg_latency_ms']:.2f} | {row['map50_95']:.4f} |\n"

report_content += """

## 3. Deployment Recommendations

### CPU Deployment: OpenVINO INT8
### GPU Deployment: TensorRT FP16/INT8
### Cross-Platform: ONNX

---

*Report generated automatically from deployment optimization pipeline.*
"""

with open(report_file, 'w') as f:
    f.write(report_content)

print(f"✓ Deployment summary report saved to: {report_file}")
print(f"\nAll outputs saved to: {output_dir}")

✓ Deployment summary report saved to: /home/tiehangz/proj/yolov12/modification/outputs/deployment_summary_report.md

All outputs saved to: /home/tiehangz/proj/yolov12/modification/outputs
